# Module 06: Monitoring & Drift Detection

**What you'll learn:**
- Why deployed models degrade over time
- What data drift is and how to detect it
- How to track model performance in production
- The monitoring feedback loop that closes the MLOps cycle

**Time:** ~1.5 hours

## 1. Why Monitoring is Critical

> "Deploying a model is not the finish line — it's the starting line."

**The scary truth**: Your model WILL get worse over time. Here's a real scenario:

1. You train an energy demand model on 2023 data (mostly normal weather)
2. You deploy it in January 2024
3. February 2024 has an unprecedented heat wave
4. Energy demand patterns completely change
5. Your model is silently giving terrible predictions
6. Nobody notices for 3 weeks
7. Bad predictions cost the utility company $2M in over-procurement

**Monitoring** catches this. Two types of problems:

| Problem | What Changes | Example |
|---------|-------------|--------|
| **Data Drift** | Input distributions | Temperature ranges shift |
| **Concept Drift** | Input→Output relationship | Same temp, different demand |

## 2. What is Data Drift?

**Data drift** = the data your model sees in production looks different from what it was trained on.

Let's see this visually:

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from energy_forecast.data.synthetic import SyntheticDataGenerator

# Reference data (what model was trained on)
gen1 = SyntheticDataGenerator(num_buildings=5, start_date='2023-01-01', end_date='2023-06-30', random_seed=42)
reference_df = gen1.generate()

# Current data (what model sees now - slightly different)
gen2 = SyntheticDataGenerator(num_buildings=5, start_date='2023-07-01', end_date='2023-12-31', random_seed=99)
current_df = gen2.generate()

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['energy_demand_kwh', 'temperature', 'humidity']):
    ax.hist(reference_df[col], bins=50, alpha=0.5, label='Training Data', density=True, color='steelblue')
    ax.hist(current_df[col], bins=50, alpha=0.5, label='Production Data', density=True, color='coral')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Distribution Comparison: Training vs Production', fontsize=14)
plt.tight_layout()
plt.show()
print('If distributions look different, your model may not be reliable on the new data!')

## 3. Statistical Drift Detection

### How do we MEASURE drift?

We use statistical tests:

- **Kolmogorov-Smirnov (KS) test**: Measures the maximum distance between two distributions. Returns a p-value — if p < 0.05, distributions are significantly different.
- **Population Stability Index (PSI)**: Quantifies how much a distribution has shifted. PSI > 0.2 = significant drift.

Let's use our DriftDetector:

In [ ]:
from energy_forecast.monitoring.drift import DriftDetector

# Create detector with reference data
detector = DriftDetector(
    reference_data=reference_df,
    config={'features_to_monitor': ['energy_demand_kwh', 'temperature', 'humidity']}
)

# Test on similar data (should show minimal drift)
report = detector.detect_drift(current_df)
print('=== Normal Data ===')
print(f'Dataset drifted: {report.is_drifted}')
print(f'Overall drift score: {report.dataset_drift_score:.4f}')
for feature, result in report.feature_drift.items():
    print(f'  {feature}: drifted={result.is_drifted}, score={result.drift_score:.4f}, p-value={result.p_value:.4f}')

### Now let's introduce REAL drift

Simulate a scenario where temperature increases dramatically:

In [ ]:
# Create drifted data
drifted_df = current_df.copy()
drifted_df['temperature'] = drifted_df['temperature'] + 15  # Heat wave!
drifted_df['energy_demand_kwh'] = drifted_df['energy_demand_kwh'] * 1.5  # Higher demand

report = detector.detect_drift(drifted_df)
print('=== Drifted Data (simulated heat wave) ===')
print(f'Dataset drifted: {report.is_drifted}')
print(f'Overall drift score: {report.dataset_drift_score:.4f}')
for feature, result in report.feature_drift.items():
    status = 'DRIFTED' if result.is_drifted else 'OK'
    print(f'  {feature}: [{status}] score={result.drift_score:.4f}, p-value={result.p_value:.6f}')

print('\nThe detector caught the drift! Time to retrain.')

## 4. Performance Monitoring

Drift tells you **inputs changed**. Performance monitoring tells you **outputs are getting worse**.

Let's simulate a model that degrades over time:

In [ ]:
from energy_forecast.monitoring.performance import PerformanceMonitor
from datetime import datetime, timedelta

monitor = PerformanceMonitor(config={
    'metrics': ['mae', 'rmse', 'mape', 'r2'],
    'degradation_threshold': 0.15
})

# Simulate 14 days of predictions with increasing noise (degrading model)
for day in range(14):
    y_true = np.random.uniform(30, 100, 24)
    noise = np.random.normal(0, 3 + day * 0.5, 24)  # Noise grows each day!
    y_pred = y_true + noise
    ts = datetime(2023, 7, 1) + timedelta(days=day)
    monitor.track_prediction(y_true, y_pred, timestamp=ts)

history = monitor.get_performance_history()
print('Performance over 14 days:')
print(history[['mae', 'rmse', 'r2']].to_string())

# Plot degradation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
history['rmse'].plot(ax=axes[0], marker='o', color='coral')
axes[0].set_title('RMSE Over Time (higher = worse)')
axes[0].set_ylabel('RMSE')
history['r2'].plot(ax=axes[1], marker='o', color='steelblue')
axes[1].set_title('R² Over Time (lower = worse)')
axes[1].set_ylabel('R²')
plt.tight_layout()
plt.show()

## 5. Degradation Detection

Set a baseline (from initial deployment) and detect when performance drops:

In [ ]:
baseline_metrics = {'mae': 3.0, 'rmse': 4.0, 'mape': 5.0, 'r2': 0.95}
current = monitor.compute_metrics(window_days=7)

degradation = monitor.detect_degradation(baseline_metrics, current)
print(f'Degraded: {degradation.is_degraded}')
print(f'Message: {degradation.alert_message}')
print(f'\nMetric deltas (positive = worse):')
for metric, delta in degradation.metric_deltas.items():
    direction = 'worse' if delta > 0 else 'better'
    print(f'  {metric}: {delta:+.1%} ({direction})')

## 6. Prometheus & Grafana

**Prometheus** collects metrics from your running services every 15 seconds:
- Prediction latency
- Request counts
- Drift scores
- Model status

**Grafana** visualizes these metrics in real-time dashboards.

With Docker Compose, both are pre-configured:
- Prometheus: http://localhost:9090
- Grafana: http://localhost:3000 (admin/admin)

Our pre-built dashboards show:
- API latency (p50/p95/p99)
- Prediction throughput
- Drift scores per feature
- Model performance over time

## 7. The Monitoring Feedback Loop

This is the complete picture that closes the MLOps cycle:

```
1. Model serves predictions → logs to Prometheus
2. Drift detector runs daily → computes drift scores
3. Performance monitor checks accuracy → detects degradation
4. If degraded → send alert → trigger retraining pipeline
5. New model is trained → evaluated → promoted to Production
6. Cycle repeats
```

The system **monitors itself and heals itself**. This is what separates a hobby project from production ML.

## 8. Exercises

1. **Drift sensitivity**: Gradually increase temperature drift (by 5, 10, 15, 20 degrees). At what point does the detector trigger?
2. **Custom threshold**: Set `degradation_threshold` to 0.05 instead of 0.15. How does this affect alert sensitivity?
3. **Visualization**: Plot drift scores over time as you gradually introduce drift.

## 9. Key Takeaways

- Models **degrade silently** — monitoring catches problems before users notice
- **Data drift** (inputs change) and **concept drift** (relationships change) are different problems
- **Statistical tests** (KS, PSI) quantify distribution changes mathematically
- **Performance monitoring** tracks prediction accuracy over time
- The **monitoring feedback loop** (detect → alert → retrain → deploy) is the heart of MLOps

**Next: [Notebook 07 - CI/CD & Deployment](./07_cicd_and_deployment.ipynb)**